# WAHIS + FAOSTAT Event Pipeline

This notebook filters WAHIS to HxNx influenza events, aggregates outbreaks to one row per event, matches the nearest FAOSTAT poultry-production snapshot by cleaned country name plus nearest year, and writes the merged CSV to `output/wahis_hxnx_event_level_with_faostat.csv`.

In [ ]:
from pathlib import Path
from difflib import get_close_matches
import re
import unicodedata

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WAHIS_PATH = ROOT / 'data' / 'WAHIS-2026-03-31.csv'
FAOSTAT_PATH = ROOT / 'data' / 'FAOSTAT_data_en_3-28-2026.csv'
OUTPUT_PATH = ROOT / 'output' / 'wahis_hxnx_event_level_with_faostat.csv'

WAHIS_PATH, FAOSTAT_PATH, OUTPUT_PATH

(PosixPath('/home/azureuser/5123_project_2/data/WAHIS-2026-03-31.csv'),
 PosixPath('/home/azureuser/5123_project_2/data/FAOSTAT_data_en_3-28-2026.csv'),
 PosixPath('/home/azureuser/5123_project_2/output/wahis_hxnx_event_level_with_faostat.csv'))

In [ ]:
HXNX_PATTERN = re.compile(r'^H\d+(?:N\d+| \(N untyped\))?$')

COUNTRY_OVERRIDES = {
    'Central African (Rep.)': 'central african republic',
    "China (People's Rep. of)": 'china mainland',
    'Chinese Taipei': 'china taiwan province of',
    'Congo (Dem. Rep. of the)': 'democratic republic of the congo',
    'Congo (Rep. of the)': 'congo',
    "Cote D'Ivoire": 'cote divoire',
    'Czech Republic': 'czechia',
    'Dominican (Rep.)': 'dominican republic',
    'Former Yug. Rep. of Macedonia': 'north macedonia',
    'Hong Kong': 'china hong kong sar',
    "Korea (Dem People's Rep. of)": 'democratic peoples republic of korea',
    'Korea (Rep. of)': 'republic of korea',
    'Laos': 'lao peoples democratic republic',
    'Netherlands': 'netherlands kingdom of the',
    'Palestinian Territory': 'palestine',
    'Reunion': 'reunion',
    'Russia': 'russian federation',
    'South Sudan (Rep. of)': 'south sudan',
    'St. Lucia': 'saint lucia',
    'Syria': 'syrian arab republic',
    'Tanzania': 'united republic of tanzania',
    'Turkey': 'turkiye',
    'Turkiye (Rep. of)': 'turkiye',
    'United Kingdom': 'united kingdom of great britain and northern ireland',
    'Vietnam': 'viet nam',
    'Brunei': 'brunei darussalam',
    'Bolivia': 'bolivia plurinational state of',
    'Iran': 'iran islamic republic of',
    'Moldova': 'republic of moldova',
    'Swaziland': 'eswatini',
    'turkiye rep of': 'turkiye',
    'Venezuela': 'venezuela bolivarian republic of',
}


def normalize_ascii(value):
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    text = text.lower().strip()
    for old, new in {
        '&': ' and ',
        "'": '',
        '.': ' ',
        ',': ' ',
        '-': ' ',
        '(': ' ',
        ')': ' ',
        '/': ' ',
    }.items():
        text = text.replace(old, new)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_feature_name(value):
    text = normalize_ascii(value)
    if pd.isna(text):
        return 'missing'
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')


def first_valid(series):
    for value in series:
        if pd.isna(value):
            continue
        if isinstance(value, str) and not value.strip():
            continue
        return value
    return pd.NA


def unique_join(series, sep=' | '):
    values = []
    seen = set()
    for value in series:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text.lower() == 'nan':
            continue
        if text not in seen:
            seen.add(text)
            values.append(text)
    return sep.join(values) if values else pd.NA


def season_from_date(value):
    if pd.isna(value):
        return pd.NA
    month = value.month
    if month in (12, 1, 2):
        return 'Winter'
    if month in (3, 4, 5):
        return 'Spring'
    if month in (6, 7, 8):
        return 'Summer'
    return 'Autumn'


def resolve_faostat_country(country, country_lookup):
    if pd.isna(country):
        return pd.Series({'faostat_country': pd.NA, 'faostat_country_match_type': pd.NA})

    direct_key = normalize_ascii(country)
    override_key = COUNTRY_OVERRIDES.get(str(country)) or COUNTRY_OVERRIDES.get(direct_key)

    if direct_key in country_lookup:
        return pd.Series(
            {
                'faostat_country': country_lookup[direct_key],
                'faostat_country_match_type': 'normalized_exact',
            }
        )

    if override_key in country_lookup:
        return pd.Series(
            {
                'faostat_country': country_lookup[override_key],
                'faostat_country_match_type': 'manual_override',
            }
        )

    search_key = override_key or direct_key
    close = get_close_matches(search_key, list(country_lookup.keys()), n=1, cutoff=0.88)
    if close:
        return pd.Series(
            {
                'faostat_country': country_lookup[close[0]],
                'faostat_country_match_type': 'fuzzy_closest',
            }
        )

    return pd.Series({'faostat_country': pd.NA, 'faostat_country_match_type': 'unmatched'})


In [ ]:
wahis = pd.read_csv(WAHIS_PATH, low_memory=False)

date_columns = [
    'event_started_on',
    'event_confirmed_on',
    'event_last_occurrence',
    'event_ended_on',
    'reported_on',
    'report_created_at',
    'start_date',
    'end_date',
    'lab_result_date',
]

for column in date_columns:
    if column in wahis.columns:
        wahis[column] = pd.to_datetime(wahis[column], errors='coerce', utc=True)

hxnx_mask = (
    wahis['disease_name'].fillna('').str.contains('influenza', case=False)
    & wahis['subtype_disease_name'].fillna('').astype(str).str.strip().str.match(HXNX_PATTERN)
)

wahis_hxnx = wahis.loc[hxnx_mask].copy()
wahis_hxnx['event_created_at'] = (
    wahis_hxnx['report_created_at']
    .combine_first(wahis_hxnx['event_started_on'])
    .combine_first(wahis_hxnx['start_date'])
)
wahis_hxnx['event_anchor_year'] = wahis_hxnx['event_created_at'].dt.year.astype('Int64')

print(
    {
        'wahis_rows': int(len(wahis)),
        'hxnx_rows': int(len(wahis_hxnx)),
        'hxnx_events': int(wahis_hxnx['event_id'].nunique()),
        'hxnx_countries': int(wahis_hxnx['country'].nunique()),
    }
)
wahis_hxnx[['event_id', 'country', 'subtype_disease_name', 'event_created_at']].head()

{'wahis_rows': 183720, 'hxnx_rows': 47118, 'hxnx_events': 2098, 'hxnx_countries': 138}


,event_id,country,subtype_disease_name,event_created_at
0,7191,Sweden,H5N2,2026-03-30 13:55:44.950000+00:00
1,6821,Austria,H5N1,2026-03-30 13:01:29.078000+00:00
2,6821,Austria,H5N1,2026-03-30 13:01:29.078000+00:00
3,6821,Austria,H5N1,2026-03-30 13:01:29.078000+00:00
4,6821,Austria,H5N1,2026-03-30 13:01:29.078000+00:00


In [ ]:
outbreak_level = (
    wahis_hxnx.sort_values(['event_id', 'outbreak_id', 'report_created_at'])
    .groupby(['event_id', 'outbreak_id'], dropna=False)
    .agg(
        outbreak_start_date=('start_date', 'min'),
        outbreak_end_date=('end_date', 'max'),
        cluster_count=('cluster_count', 'max'),
        quant_total_entry_count=('quant_total_entry_count', 'max'),
        admin_divisions=('admin_division', unique_join),
        locations=('location', unique_join),
        epi_unit_types=('epi_unit_type', unique_join),
        location_approx_any=('location_approx', 'max'),
        mean_longitude=('longitude', 'mean'),
        mean_latitude=('latitude', 'mean'),
        outbreak_descriptions=('outbreak_description', unique_join),
        source_names=('source_names', unique_join),
        lab_species=('lab_species', unique_join),
        lab_test_names=('lab_test_name', unique_join),
        lab_names=('lab_name', unique_join),
        lab_results=('lab_result', unique_join),
        lab_result_date=('lab_result_date', 'max'),
    )
    .reset_index()
)

outbreak_level['outbreak_reference_end'] = outbreak_level['outbreak_end_date'].combine_first(
    outbreak_level['outbreak_start_date']
)

event_static = (
    wahis_hxnx.sort_values(['event_id', 'report_created_at'])
    .groupby('event_id', dropna=False)
    .agg(
        country=('country', first_valid),
        country_iso=('country_iso', first_valid),
        disease_id=('disease_id', first_valid),
        disease_name=('disease_name', first_valid),
        subtype_disease_id=('subtype_disease_id', first_valid),
        subtype_disease_name=('subtype_disease_name', unique_join),
        event_started_on=('event_started_on', 'min'),
        event_confirmed_on=('event_confirmed_on', 'min'),
        event_last_occurrence=('event_last_occurrence', 'max'),
        event_ended_on=('event_ended_on', 'max'),
        event_status=('event_status', first_valid),
        reason=('reason', unique_join),
        first_reported_on=('reported_on', 'min'),
        last_reported_on=('reported_on', 'max'),
        first_report_created_at=('report_created_at', 'min'),
        last_report_created_at=('report_created_at', 'max'),
        report_numbers=('report_number', unique_join),
        report_statuses=('report_status', unique_join),
        report_count=('report_id', 'nunique'),
    )
    .reset_index()
)

event_static['event_created_at'] = event_static['first_report_created_at'].combine_first(
    event_static['event_started_on']
)
event_static['event_anchor_year'] = event_static['event_created_at'].dt.year.astype('Int64')

event_last_snapshot = (
    wahis_hxnx.sort_values(
        ['event_id', 'report_created_at', 'reported_on', 'start_date', 'outbreak_id'],
        na_position='last',
    )
    .groupby('event_id', dropna=False)
    .tail(1)
    .copy()
)

last_flock_lost_columns = [
    'event_id',
    'quant_total_unit',
    'quant_total_entry_count',
    'quant_total_1_species_name',
    'quant_total_1_deaths_killed_slaughtered_total',
    'quant_total_2_species_name',
    'quant_total_2_deaths_killed_slaughtered_total',
    'quant_total_3_species_name',
    'quant_total_3_deaths_killed_slaughtered_total',
]

event_last_snapshot = event_last_snapshot[last_flock_lost_columns].rename(
    columns={
        'quant_total_unit': 'last_quant_total_unit',
        'quant_total_entry_count': 'last_quant_total_entry_count',
        'quant_total_1_species_name': 'last_quant_total_1_species_name',
        'quant_total_1_deaths_killed_slaughtered_total': 'last_quant_total_1_deaths_killed_slaughtered_total',
        'quant_total_2_species_name': 'last_quant_total_2_species_name',
        'quant_total_2_deaths_killed_slaughtered_total': 'last_quant_total_2_deaths_killed_slaughtered_total',
        'quant_total_3_species_name': 'last_quant_total_3_species_name',
        'quant_total_3_deaths_killed_slaughtered_total': 'last_quant_total_3_deaths_killed_slaughtered_total',
    }
)

event_last_snapshot['last_flock_lost_total'] = event_last_snapshot[
    [
        'last_quant_total_1_deaths_killed_slaughtered_total',
        'last_quant_total_2_deaths_killed_slaughtered_total',
        'last_quant_total_3_deaths_killed_slaughtered_total',
    ]
].fillna(0).sum(axis=1)

event_outbreak = (
    outbreak_level.groupby('event_id', dropna=False)
    .agg(
        outbreak_count=('outbreak_id', lambda series: series.nunique(dropna=True)),
        first_outbreak_start=('outbreak_start_date', 'min'),
        last_outbreak_end=('outbreak_end_date', 'max'),
        last_outbreak_reference=('outbreak_reference_end', 'max'),
        cluster_count_total=('cluster_count', 'sum'),
        quant_total_entry_count_total=('quant_total_entry_count', 'sum'),
        admin_divisions=('admin_divisions', unique_join),
        locations=('locations', unique_join),
        epi_unit_types=('epi_unit_types', unique_join),
        location_approx_any=('location_approx_any', 'max'),
        mean_longitude=('mean_longitude', 'mean'),
        mean_latitude=('mean_latitude', 'mean'),
        outbreak_descriptions=('outbreak_descriptions', unique_join),
        source_names=('source_names', unique_join),
        lab_species=('lab_species', unique_join),
        lab_test_names=('lab_test_names', unique_join),
        lab_names=('lab_names', unique_join),
        lab_results=('lab_results', unique_join),
        last_lab_result_date=('lab_result_date', 'max'),
    )
    .reset_index()
)

event_level = event_static.merge(event_outbreak, on='event_id', how='left')
event_level = event_level.merge(event_last_snapshot, on='event_id', how='left')
event_level['season_started'] = event_level['first_outbreak_start'].map(season_from_date)
event_level['confirmation_lag_days'] = (
    event_level['first_outbreak_start'] - event_level['event_started_on']
).dt.days.astype('Int64')
event_level['outbreak_span_days'] = (
    event_level['last_outbreak_reference'] - event_level['first_outbreak_start']
).dt.days.astype('Int64')

event_level[['event_id', 'country', 'subtype_disease_name', 'last_flock_lost_total', 'outbreak_count', 'season_started', 'confirmation_lag_days', 'outbreak_span_days']].head()

,event_id,country,subtype_disease_name,last_flock_lost_total,outbreak_count,season_started,confirmation_lag_days,outbreak_span_days
0,4,Denmark,H5N1,102.0,1,Spring,0,13
1,5,Czech Republic,H5N1,4.0,1,Spring,7,26
2,8,Niger,H5N1,22034.0,2,Winter,0,116
3,10,Israel,H5N1,271400.0,9,Spring,0,17
4,13,Denmark,H5N3,26780.0,2,Summer,0,62


In [ ]:
faostat = pd.read_csv(FAOSTAT_PATH)
faostat['country_key'] = faostat['Area'].map(normalize_ascii)
faostat['feature_name'] = (
    'faostat_'
    + faostat['Element'].map(normalize_feature_name)
    + '__'
    + faostat['Item'].map(normalize_feature_name)
    + '__'
    + faostat['Unit'].map(normalize_feature_name)
)

faostat_wide = (
    faostat.pivot_table(
        index=['Area', 'country_key', 'Year'],
        columns='feature_name',
        values='Value',
        aggfunc='sum',
    )
    .reset_index()
)
faostat_wide.columns.name = None
faostat_wide = faostat_wide.rename(columns={'Area': 'faostat_country', 'Year': 'faostat_year'})

country_lookup = (
    faostat_wide[['country_key', 'faostat_country']]
    .drop_duplicates()
    .set_index('country_key')['faostat_country']
    .to_dict()
)

resolved_countries = [resolve_faostat_country(value, country_lookup) for value in event_level['country']]
event_level['faostat_country'] = [result['faostat_country'] for result in resolved_countries]
event_level['faostat_country_match_type'] = [result['faostat_country_match_type'] for result in resolved_countries]

faostat_feature_columns = [
    column
    for column in faostat_wide.columns
    if column not in {'faostat_country', 'country_key', 'faostat_year'}
]

events_for_match = event_level.loc[
    event_level['faostat_country'].notna() & event_level['event_anchor_year'].notna(),
    ['event_id', 'faostat_country', 'event_anchor_year'],
].copy()
events_for_match['event_anchor_year'] = events_for_match['event_anchor_year'].astype('int64')

candidate_matches = events_for_match.merge(
    faostat_wide,
    on='faostat_country',
    how='left',
)
candidate_matches['faostat_year_gap'] = (
    candidate_matches['event_anchor_year'] - candidate_matches['faostat_year'].astype('int64')
).abs().astype('Int64')
matched_events = (
    candidate_matches.sort_values(['event_id', 'faostat_year_gap', 'faostat_year'])
    .drop_duplicates(subset=['event_id'], keep='first')
)

event_level = event_level.merge(
    matched_events[['event_id', 'faostat_year', 'faostat_year_gap', *faostat_feature_columns]],
    on='event_id',
    how='left',
)

columns_to_drop = {
    column for column in event_level.columns if column.endswith('_id')
}
columns_to_drop.update(
    {
        'country',
        'country_iso',
        'faostat_country',
        'event_last_occurrence',
        'outbreak_descriptions',
        'admin_divisions'
    }
)
event_level = event_level.drop(columns=[column for column in columns_to_drop if column in event_level.columns])

high_missing_columns = event_level.columns[event_level.isna().mean() > 0.20].tolist()
event_level = event_level.drop(columns=high_missing_columns)

event_level = event_level.sort_values(['event_created_at']).reset_index(drop=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
event_level.to_csv(OUTPUT_PATH, index=False)

print({'output_rows': int(len(event_level)), 'output_columns': int(len(event_level.columns))})
print('Dropped for missingness > 20%:', high_missing_columns)
event_level.head()

{'output_rows': 2098, 'output_columns': 37}
Dropped for missingness > 20%: ['first_reported_on', 'last_reported_on', 'first_report_created_at', 'last_report_created_at', 'lab_species', 'lab_test_names', 'lab_names', 'lab_results', 'last_lab_result_date', 'last_quant_total_2_species_name', 'last_quant_total_2_deaths_killed_slaughtered_total', 'last_quant_total_3_species_name', 'last_quant_total_3_deaths_killed_slaughtered_total', 'faostat_production__meat_of_ducks_fresh_or_chilled__t', 'faostat_production__meat_of_geese_fresh_or_chilled__t']


,disease_name,subtype_disease_name,event_started_on,event_confirmed_on,event_ended_on,event_status,reason,report_numbers,report_statuses,report_count,...,last_quant_total_1_species_name,last_quant_total_1_deaths_killed_slaughtered_total,last_flock_lost_total,season_started,confirmation_lag_days,outbreak_span_days,faostat_country_match_type,faostat_year,faostat_year_gap,faostat_production__meat_of_chickens_fresh_or_chilled__t
0,Highly pathogenic avian influenza,H7N7,2005-02-25 00:00:00+00:00,2005-03-23 00:00:00+00:00,2005-03-07 00:00:00+00:00,Resolved,First occurrence,0,Validated,1,...,Birds,218788.0,218788.0,Winter,0,10,manual_override,2005.0,0,35750.0
1,Highly pathogenic avian influenza,H5N1,2005-07-18 00:00:00+00:00,2005-07-23 00:00:00+00:00,2005-12-12 00:00:00+00:00,Resolved,First occurrence,5,Validated,1,...,Birds,623364.0,624666.0,Summer,0,147,manual_override,2005.0,0,1345725.0
2,Highly pathogenic avian influenza,H5N1,2005-07-22 00:00:00+00:00,2005-07-29 00:00:00+00:00,2005-08-01 00:00:00+00:00,Resolved,First occurrence,2,Validated,1,...,Birds,2800.0,2800.0,Summer,0,10,normalized_exact,2005.0,0,45700.0
3,Highly pathogenic avian influenza,H5N1,2005-08-02 00:00:00+00:00,2005-08-07 00:00:00+00:00,2005-08-27 00:00:00+00:00,Resolved,First occurrence,2,Validated,1,...,Wildlife (species unspecified),89.0,89.0,Summer,0,25,normalized_exact,2005.0,0,196.0
4,Highly pathogenic avian influenza,H5N1,2005-10-01 00:00:00+00:00,2005-10-06 00:00:00+00:00,2005-12-08 00:00:00+00:00,Resolved,First occurrence,4,Validated,1,...,Birds,1800.0,1800.0,Autumn,0,68,manual_override,2005.0,0,936697.0


In [ ]:
column_completeness = (
    pd.DataFrame(
        {
            'column': event_level.columns,
            'non_missing_count': event_level.notna().sum().values,
            'missing_count': event_level.isna().sum().values,
            'completeness_pct': event_level.notna().mean().mul(100).round(2).values,
        }
    )
    .sort_values(['missing_count', 'column'], ascending=[False, True])
    .reset_index(drop=True)
)

complete_columns = column_completeness.loc[column_completeness['missing_count'].eq(0), 'column'].tolist()
missing_columns = column_completeness.loc[column_completeness['missing_count'].gt(0), 'column'].tolist()

print(
    {
        'complete_columns': len(complete_columns),
        'columns_with_missing_values': len(missing_columns),
    }
)
column_completeness

{'complete_columns': 17, 'columns_with_missing_values': 20}


,column,non_missing_count,missing_count,completeness_pct
0,event_ended_on,1907,191,90.90
1,last_outbreak_end,2034,64,96.95
2,last_quant_total_1_deaths_killed_slaughtered_t...,2062,36,98.28
3,source_names,2079,19,99.09
4,faostat_production__meat_of_chickens_fresh_or_...,2080,18,99.14
5,faostat_year,2081,17,99.19
6,faostat_year_gap,2081,17,99.19
7,admin_divisions,2095,3,99.86
8,confirmation_lag_days,2095,3,99.86
9,epi_unit_types,2095,3,99.86
